# 03 · 基因组数据格式 Genomics Data Formats

> **能力标签**：GB4（基因组数据格式与流水线 / Genomics Data Formats & Pipelines）

## 目标 Objectives

完成本课后，你应该能够：

1. 描述 **PLINK1 二进制格式**（`.bed/.bim/.fam`）和 **PLINK2 格式**（`.pgen/.pvar/.psam`）的结构，说明它们的区别与适用场景。
2. 解释 **BGEN 格式**（`.bgen/.bgi`）——存储**剂量概率（dosage probabilities）**的主要格式，以及为什么 UK Biobank 使用 BGEN。
3. 理解 **VCF/BCF**（Variant Call Format / Binary VCF）的基本字段结构，以及 **tabix 索引**如何支持按区域快速查询。
4. 区分**硬调用基因型（hard-call genotype）**与**剂量（dosage）**，了解剂量在不确定位点（如芯片填充后）的重要性。
5. 了解基因组坐标系统 **hg19**（GRCh37）与 **hg38**（GRCh38）的区别，以及 **liftover** 操作的含义。
6. 读懂 **REGENIE 输出格式**（sumstats 文件），识别关键列（CHROM, GENPOS, ALLELE0/1, BETA, SE, LOG10P 等）。

> 对应能力：**GB4**（本课以概念/散文为主，运行代码仅用纯 Python 演示核心思路）


## 概念讲解 Concepts

### 1 · PLINK 格式 PLINK Formats

#### 1a · PLINK1 二进制格式（.bed / .bim / .fam）

PLINK1 是 GWAS 领域沿用最广的格式之一，由三个配套文件组成：

| 文件 | 内容 | 说明 |
|------|------|------|
| `.bed` | 二进制基因型矩阵 | 每个样本-SNP 对用 2 bit 编码（00=hom ref, 01=missing, 10=het, 11=hom alt） |
| `.bim` | 变异注释 | 文本，每行一个 SNP：染色体、变异 ID、遗传距离（cM）、位置（bp）、等位基因 A1/A2 |
| `.fam` | 样本信息 | 文本，每行一个个体：家系 ID、个体 ID、父/母 ID、性别、表型 |

`.bed` 文件以**变异优先（variant-major / SNP-major）**顺序存储，每个 SNP 占用一个连续的 $\lceil n/4 \rceil$ 字节块（4 个基因型 per byte），顺序为所有样本在该 SNP 上的基因型。

**局限**：
- 只存储硬调用基因型（hard-call），无法表示填充概率
- PLINK2 增强了随机变异访问能力，同时支持剂量和压缩

---

#### 1b · PLINK2 格式（.pgen / .pvar / .psam）

PLINK2（Chang et al.）重新设计了二进制格式：

| 文件 | 内容 |
|------|------|
| `.pgen` | 新型压缩基因型，支持缺失、多等位基因、剂量概率 |
| `.pvar` | 变异注释（类 VCF 头，tab 分隔） |
| `.psam` | 样本信息（替代 .fam，支持更多字段） |

**优势**：
- **变异优先（variant-major）**访问：随机访问单个变异更快
- 支持存储**剂量（dosage）**和**填充概率（imputation probabilities）**
- 更高压缩率，支持多等位基因变异（multi-allelic）

`pgenlib` 是 Python 读取 `.pgen` 的官方库（已在本环境安装），
可通过 `pgenlib.PgenReader` 按变异块批量读取基因型矩阵。

---

### 2 · BGEN 格式 BGEN Format

**BGEN**（.bgen + .bgi 索引）是 **UK Biobank** 发布填充基因型时使用的标准格式，
设计初衷是存储**不确定（imputed）的概率剂量**，而非简单的整数基因型。

**核心数据结构**：
- 每个位点、每个样本存储三个概率：$P(\text{hom ref}) + P(\text{het}) + P(\text{hom alt}) = 1$
- 期望剂量（expected dosage）= $0 \times P_{\text{hom ref}} + 1 \times P_{\text{het}} + 2 \times P_{\text{hom alt}} \in [0, 2]$

**为什么需要 BGEN？**
芯片基因分型（chip genotyping）只覆盖约 50 万–100 万个 SNP，
而通过**填充（imputation）**（参考 1000 Genomes / HRC / TOPMed 等面板）可以推断多达数千万个 SNP 的概率剂量。
这些填充后的位点携带不确定性（信息量 INFO score 可能 < 1），最好用概率来表示。

**BGEN 版本**：BGEN v1.2/v1.3 支持可变精度的概率存储（8/16/32 bit）。

**常用工具**：`bgenix`（索引）、`qctool`（转换）、`rbgen`（R 包）、Python 下的 `bgen_reader`。

---

### 3 · VCF/BCF 格式 VCF/BCF Format

**VCF（Variant Call Format）** 是变异发现（variant calling，如 WGS/WES）的通用文本格式：

```
##fileformat=VCFv4.2
##INFO=<ID=AF,Number=A,Type=Float,Description="Allele frequency">
#CHROM  POS     ID      REF  ALT  QUAL  FILTER  INFO          FORMAT  SAMPLE1 SAMPLE2
1       752566  rs3094315  A   G    .     PASS   AF=0.23       GT:GP   0/1     0/0
```

**关键字段**：
- `CHROM`：染色体
- `POS`：基于 1 的位置（1-based coordinate）
- `REF` / `ALT`：参考和替代等位基因
- `FORMAT`：每个样本的字段格式（GT=基因型, GP=基因型概率, DS=剂量）
- `GT`：基因型（0/0, 0/1, 1/1 分别对应剂量 0, 1, 2）

**BCF**（Binary VCF）：VCF 的二进制压缩版，通过 **htslib** 工具包（`bgzip`、`bcftools`）处理，
文件更小，随机访问更快。

---

### 4 · Tabix 索引 Tabix Indexing

**tabix** 是 htslib 的配套工具，对 bgzip 压缩的 VCF/BED/TSV 建立坐标索引（`.tbi` 文件），
支持按染色体区域（region）快速查询：

```bash
bgzip variants.vcf             # bgzip 压缩
tabix -p vcf variants.vcf.gz   # 建立 .tbi 索引
tabix variants.vcf.gz 1:1000000-2000000   # 查询 chr1 的某区段
```

**工作原理**：将 bgzip 的压缩块（BGZF block）的起始偏移量（virtual offset）按区间索引，
查询时直接 seek 到对应块，无需扫描整个文件。

---

### 5 · 剂量 vs 硬调用 Dosage vs Hard-Call

| 概念 | 定义 | 使用场景 |
|------|------|---------|
| **硬调用（hard-call）** | $\{0, 1, 2\}$ 整数，取概率最高的基因型 | 简单分析、LD 计算 |
| **剂量（dosage）** | $[0, 2]$ 连续值，$= E[\text{ALT 拷贝数}]$ | 填充位点的关联测试，保留不确定性 |

在关联测试中使用剂量（dosage）比硬调用有更高的统计效能（power），
因为它保留了填充不确定性信息，避免了因四舍五入引入的信息损失。

---

### 6 · 基因组坐标系 hg19 vs hg38

人类参考基因组有两个主要版本：

| 版本 | 别名 | 发布年份 | 特点 |
|------|------|---------|------|
| **hg19** | GRCh37 | 2009 | 历史更悠久，大量 GWAS 数据基于此版本 |
| **hg38** | GRCh38 | 2013 | 更完整，修正了许多 hg19 的错误，现为推荐版本 |

同一个变异在 hg19 和 hg38 中的**坐标（position）不同**！
因为 hg38 对某些区域的组装进行了修改，导致下游坐标平移。

**Liftover**：将变异坐标从一个基因组版本转换到另一个版本。
常用工具：UCSC `liftOver`（.chain 文件），Picard `LiftoverVcf`，Python `pyliftover`。

**注意**：liftover 过程中约 1–2% 的变异可能因区域重组、倒位等原因无法成功转换（unmapped）。

---

### 7 · REGENIE 输出格式 REGENIE Output

REGENIE 完成 Step 2 后，对每个表型输出一个制表符分隔的汇总统计文件（sumstats），
常见列包括：

| 列名 | 含义 |
|------|------|
| `CHROM` | 染色体 |
| `GENPOS` | 基因组位置（bp） |
| `ID` | 变异 ID（rsID 或自定义） |
| `ALLELE0` | 参考等位基因（通常为 REF） |
| `ALLELE1` | 效应等位基因（通常为 ALT） |
| `A1FREQ` | 效应等位基因频率 |
| `N` | 样本量 |
| `BETA` | 效应量（线性回归）或 log-OR（逻辑回归） |
| `SE` | 标准误 |
| `CHISQ` | Wald 统计量 = $(\beta/SE)^2$ |
| `LOG10P` | $-\log_{10}(p)$（比 `P` 列在极小值时精度更高） |
| `EXTRA` | 额外信息（仅二元表型） |

**LOG10P** vs **P**：当 $p$ 极小（如 $10^{-100}$）时，浮点数精度有限，
直接存储 `LOG10P` 可保留更多有效位数。


## 示例 Worked Example

以下代码演示**不依赖专用格式库**的核心概念：剂量计算、VCF 字段解析、liftover 偏移思路。

In [ ]:
import numpy as np
import tempfile, os

tmpdir = tempfile.mkdtemp()
print("演示1：剂量（dosage）计算")
print("=" * 45)

# 模拟 BGEN 风格的基因型概率（n=5 个样本，1 个 SNP）
# 每行：[P(hom_ref), P(het), P(hom_alt)]，三列之和 = 1
gp = np.array([
    [0.90, 0.08, 0.02],   # 大概率 hom ref，dosage ≈ 0.12
    [0.05, 0.88, 0.07],   # 大概率 het，dosage ≈ 1.02
    [0.03, 0.15, 0.82],   # 大概率 hom alt，dosage ≈ 1.79
    [0.50, 0.48, 0.02],   # 不确定（hom ref 或 het），dosage ≈ 0.52
    [0.01, 0.01, 0.98],   # 几乎确定 hom alt，dosage ≈ 1.97
])

# 期望剂量 = 0*P0 + 1*P1 + 2*P2
dosage = gp[:, 1] + 2 * gp[:, 2]

# 硬调用 = argmax（取概率最高的基因型）
hard_call = np.argmax(gp, axis=1)  # 0, 1, 或 2

print(f"{'样本':<6} {'P(0/0)':<8} {'P(0/1)':<8} {'P(1/1)':<8} {'剂量':<8} {'硬调用'}")
for i in range(len(dosage)):
    print(f"  {i+1:<4} {gp[i,0]:<8.2f} {gp[i,1]:<8.2f} {gp[i,2]:<8.2f} "
          f"{dosage[i]:<8.4f} {hard_call[i]}")

print()
print(f"剂量均值（= 2 * 估计 AAF）: {dosage.mean():.4f}")
print(f"硬调用的整数均值:           {hard_call.mean():.4f}")
print(f"最大剂量-硬调用差异:         {np.abs(dosage - hard_call).max():.4f}")
print("（样本 4：硬调用=0，但实际剂量=0.52，不确定性被保留在剂量中）")


In [ ]:
print("演示2：VCF 字段解析（最小化示例）")
print("=" * 45)

# 模拟一段 VCF 内容（仅内存中，不读取文件）
vcf_lines = [
    "##fileformat=VCFv4.2",
    "#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\tSAMPLE1\tSAMPLE2\tSAMPLE3",
    "1\t752566\trs3094315\tA\tG\t.\tPASS\tAF=0.23;AN=2000\tGT\t0/0\t0/1\t1/1",
    "1\t1008567\trs28765502\tT\tC\t.\tPASS\tAF=0.08;AN=2000\tGT\t0/0\t0/0\t0/1",
    "2\t234567\trs7592785\tC\tA\t.\tPASS\tAF=0.45;AN=2000\tGT\t1/1\t0/1\t0/0",
]

print("原始 VCF 行数（含头）：", len(vcf_lines))

# 解析变异行（跳过 ## 和 # 开头的头行）
variants = []
for line in vcf_lines:
    if line.startswith("#"):
        continue
    fields = line.split("\t")
    chrom = fields[0]
    pos   = int(fields[1])
    vid   = fields[2]
    ref   = fields[3]
    alt   = fields[4]
    # 解析 INFO 字段
    info_dict = dict(kv.split("=") for kv in fields[7].split(";") if "=" in kv)
    af = float(info_dict.get("AF", "NA"))
    # 解析样本基因型 -> 剂量
    genotypes = []
    for samp in fields[9:]:
        gt = samp.split(":")[0]
        alleles = gt.replace("|", "/").split("/")
        genotypes.append(sum(int(a) for a in alleles if a != "."))
    variants.append({
        "CHROM": chrom, "POS": pos, "ID": vid,
        "REF": ref, "ALT": alt, "AF": af,
        "dosages": genotypes,
    })

print(f"\n解析到 {len(variants)} 个变异：")
for v in variants:
    print(f"  {v['CHROM']}:{v['POS']}  {v['ID']}  {v['REF']}->{v['ALT']}"
          f"  AF={v['AF']:.2f}  样本剂量={v['dosages']}")


In [ ]:
print("演示3：坐标系统差异（hg19 vs hg38 概念示意）")
print("=" * 45)

# 假设一组 hg19 位置及其对应 hg38 位置（来自 chain 文件 liftover 结果）
# 这里用固定偏移量模拟简单的线性区域（真实 liftover 更复杂）
hg19_positions = np.array([752566, 1008567, 1234567, 3456789, 10000000])

# 模拟 liftover 映射（真实情况用 pyliftover 或 UCSC liftOver 工具）
# 在 chr1 的这个区域，hg38 相对 hg19 约有 +110 bp 的偏移（仅示意）
offset_simulated = 110
hg38_positions = hg19_positions + offset_simulated
unmapped_mask = np.array([False, False, False, True, False])  # 模拟 1 个位点无法 liftover

print(f"{'变异':<6} {'hg19 POS':<12} {'hg38 POS':<12} {'状态'}")
for i, (h19, h38, unmap) in enumerate(zip(hg19_positions, hg38_positions, unmapped_mask)):
    status = "UNMAPPED" if unmap else "OK"
    h38_str = "N/A" if unmap else str(h38)
    print(f"  SNP{i+1:<3} {h19:<12} {h38_str:<12} {status}")

n_ok = (~unmapped_mask).sum()
n_fail = unmapped_mask.sum()
print(f"\nLiftover 成功: {n_ok}/{len(hg19_positions)} ({n_ok/len(hg19_positions)*100:.0f}%)")
print(f"Liftover 失败: {n_fail}/{len(hg19_positions)} ({n_fail/len(hg19_positions)*100:.0f}%)")
print("（真实数据中约 1-2% 位点无法成功从 hg19 liftover 到 hg38）")


In [ ]:
print("演示4：REGENIE 输出格式（模拟汇总统计）")
print("=" * 45)

import io

# 模拟 REGENIE sumstats 文件内容
regenie_lines = """CHROM GENPOS ID ALLELE0 ALLELE1 A1FREQ N BETA SE CHISQ LOG10P
1 752566 rs3094315 A G 0.2300 450000 0.0312 0.0041 57.9 13.24
1 1008567 rs28765502 T C 0.0821 450000 -0.0085 0.0062 1.88 0.82
2 234567 rs7592785 C A 0.4512 449800 0.1234 0.0038 105.6 23.14
22 16050654 rs587697622 G A 0.0105 450000 0.0412 0.0151 7.46 2.89
""".strip()

# 解析
header, *data_lines = regenie_lines.split("\n")
cols = header.split()
print(f"REGENIE 输出列：{cols}")
print()

rows = []
for line in data_lines:
    row = dict(zip(cols, line.split()))
    rows.append(row)

print(f"{'CHROM':<6} {'GENPOS':<10} {'ID':<14} {'ALLELE0/1':<10} {'BETA':<8} {'SE':<8} {'LOG10P':<8} {'含义'}")
for r in rows:
    p_approx = 10 ** (-float(r["LOG10P"]))
    alleles = f"{r['ALLELE0']}/{r['ALLELE1']}"
    sig = " *GWS*" if float(r["LOG10P"]) > 7.3 else ""
    print(f"  {r['CHROM']:<4} {r['GENPOS']:<10} {r['ID']:<14} {alleles:<10} "
          f"{float(r['BETA']):<8.4f} {float(r['SE']):<8.4f} {float(r['LOG10P']):<8.2f}{sig}")

print()
print("* GWS = Genome-Wide Significant (LOG10P > 7.3 <=> p < 5e-8)")
print()
print("LOG10P 与 CHISQ 的关系验证：")
for r in rows:
    chisq = float(r["CHISQ"])
    log10p_from_chisq = -np.log10(
        float(np.format_float_scientific(
            # 用 scipy 计算的精确 p 值
            __import__("scipy.stats", fromlist=["chi2"]).chi2.sf(chisq, df=1),
            precision=6
        ))
    )
    print(f"  {r['ID']}: CHISQ={chisq:.2f}  LOG10P(储存)={r['LOG10P']}"
          f"  LOG10P(重算)={log10p_from_chisq:.2f}")


In [ ]:
print("演示5：PGEN 格式概念（pgenlib 已安装，此处只做概念说明）")
print("=" * 45)

# pgenlib 已安装，但无实际 .pgen 文件，故只演示 API 概念
# 以下代码展示如何用 pgenlib 读取 pgen（如果文件存在）

pgenlib_api_demo = '''
import pgenlib
import numpy as np

# 打开 pgen 文件（.pvar/.psam 同名前缀）
reader = pgenlib.PgenReader(b"data/chrom1.pgen")

n_samples = reader.get_raw_sample_ct()  # 样本数
n_variants = reader.get_variant_ct()    # 变异数
print(f"样本数: {n_samples}, 变异数: {n_variants}")

# 读取第 0..99 个变异的基因型（hard-call）
buf = np.empty((100, n_samples), dtype=np.int32)
reader.read_range(0, 100, buf)
# buf[i, j] 为第 i 个变异第 j 个样本的剂量（0/1/2/-9=缺失）

# 关闭
reader.close()
'''

print("pgenlib 读取 pgen 文件的 API（概念演示，不执行）：")
print(pgenlib_api_demo)

# 实际验证：检查 pgenlib 可以导入
try:
    import pgenlib
    print("pgenlib 成功导入，版本信息：", end="")
    print(getattr(pgenlib, "__version__", "（无 __version__ 属性）"))
except ImportError as e:
    print(f"pgenlib 未安装：{e}")


## 动手 Exercise

本课以概念/散文为主（GB4 没有专用自动评分题包）。请结合上述示例完成以下思考与实践：

**思考题 1**：BGEN v1.2 格式中，若以 16 bit 精度存储三个基因型概率，
且每个样本存储 2 个概率（第三个由 $1 - p_0 - p_1$ 推导），
请估算一个 22 染色体、共 1 千万个填充变异、50 万个样本的数据集的大致存储体积（GB）。

**思考题 2**：为什么 REGENIE 使用 `LOG10P` 列而非 `P` 列？
当 $p = 10^{-300}$ 时，64 位浮点数（IEEE 754 double）能否准确表示？

**思考题 3**：下面的 liftover 情景中，哪些变异**最可能**无法成功 liftover？
（a）位于基因组重复区域（segmental duplication）的变异
（b）位于常染色体唯一序列中的变异
（c）靠近着丝粒（centromere）或端粒（telomere）的变异

**编程练习**：扩展"演示2"中的 VCF 解析代码，
增加对 `FORMAT` 字段中 `DS`（dosage）的解析：
若格式为 `GT:DS`，则提取 DS 值；若仅有 `GT`，则从 0/1/1 等硬调用计算整数剂量。
将结果用字典列表返回，每项包含 `ID`、`POS`、`hard_call`（列表）、`dosage`（列表）。

```python
vcf_lines_ex = [
    "#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\tS1\tS2\tS3",
    "1\t100\trs1\tA\tT\t.\tPASS\t.\tGT:DS\t0/1:0.97\t0/0:0.03\t1/1:1.98",
    "1\t200\trs2\tG\tC\t.\tPASS\t.\tGT\t0/0\t0/1\t1/1",
]
# 你的代码写在这里
```


## 延伸阅读 Further Reading

1. **Mbatchou et al. (2021)**. "Computationally efficient whole-genome regression for quantitative and binary traits." *Nature Genetics.* REGENIE 原文，含 Step 1/Step 2 输出格式说明。
2. **Purcell et al. (2007)**. "PLINK: A tool set for whole-genome association and population-based linkage analyses." *American Journal of Human Genetics.* PLINK1 格式原始论文。
3. **Bycroft et al. (2018)**. "The UK Biobank resource with deep phenotyping and genomic data." *Nature.* UK Biobank 数据集发布，含 BGEN 格式选择的背景。
4. BGEN 格式规范：[https://www.well.ox.ac.uk/~gav/bgen_format/](https://www.well.ox.ac.uk/~gav/bgen_format/)（可离线阅读 PDF）
5. PLINK2 格式规范：`plink2 --help` 或 [https://www.cog-genomics.org/plink/2.0/](https://www.cog-genomics.org/plink/2.0/)
6. **1000 Genomes Project Consortium (2015)**. "A global reference for human genetic variation." *Nature.* 填充参考面板（imputation reference panel）背景。
7. VCF 规范（v4.3）：GATK HaplotypeCaller / bcftools 文档中的 FORMAT/INFO 字段定义。


## 关联练习 Related Assignments

本课（GB4 概念课）无独立的自动评分题包。请复习以下 W8 编程作业，
巩固 Notebook 01 和 Notebook 02 中涉及的函数：

### 复习 b8-genopca

```bash
python check.py 01-genopca
```

### 复习 b8-ridge-lmm

```bash
python check.py 02-ridge-lmm
```

### 复习 b8-loco

```bash
python check.py 03-loco
```

> 三个题包合计覆盖 GB3 能力的核心函数。在理解了本课的数据格式背景后，
> 回顾这些函数在真实 REGENIE 流水线中的作用会更加清晰。
